# CRISP-DM Fraud Detection Notebook
Predicting `is_fraud` from `shop.db`

## 1. Business Understanding
Goal: Predict fraudulent orders to prioritize review.
Success: High recall for fraud while maintaining reasonable precision.

In [2]:
import sqlite3
from pathlib import Path
import pandas as pd

from functions import basic_wrangling
from functions import unistats
from functions import parse_date
from functions import manage_dates

conn = sqlite3.connect(r"../shop.db/shop.db")

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

df = pd.read_sql("SELECT * FROM orders", conn)

# Line-level aggregates (orders table alone has no num_items / total_value)
items_agg = pd.read_sql(
    """
    SELECT order_id, SUM(quantity) AS num_items
    FROM order_items
    GROUP BY order_id
    """,
    conn,
)
df = df.merge(items_agg, on="order_id", how="left")
df["num_items"] = df["num_items"].fillna(0)

# Same meaning as ML pipeline feature name
df["total_value"] = df["order_total"]

cust = pd.read_sql("SELECT customer_id, birthdate FROM customers", conn)
df = df.merge(cust, on="customer_id", how="left")
df["order_datetime"] = pd.to_datetime(df["order_datetime"], errors="coerce")
df["birthdate"] = pd.to_datetime(df["birthdate"], errors="coerce")
df["customer_age"] = ((df["order_datetime"] - df["birthdate"]).dt.days // 365).astype("Int64")

df = df.sort_values(["customer_id", "order_datetime", "order_id"])
df["customer_order_count"] = df.groupby("customer_id").cumcount()

df.head()

              name
0        customers
1  sqlite_sequence
2         products
3           orders
4      order_items
5        shipments
6  product_reviews


,order_id,customer_id,order_datetime,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,...,shipping_fee,tax_amount,order_total,risk_score,is_fraud,num_items,total_value,birthdate,customer_age,customer_order_count
592,593,1,2025-07-09 22:19:28,28289,28289,CO,crypto,mobile,US,0,...,8.04,36.06,514.57,74.5,0,5,514.57,2005-06-08,20,0
912,913,1,2025-07-09 23:46:32,28289,28289,CO,paypal,mobile,US,0,...,8.04,47.95,574.14,37.1,0,5,574.14,2005-06-08,20,1
902,903,1,2025-07-10 02:53:00,28289,28289,CO,paypal,mobile,US,0,...,24.99,9.40,187.01,7.8,0,2,187.01,2005-06-08,20,2
343,344,1,2025-07-10 08:20:13,28289,28289,CO,card,desktop,US,0,...,8.04,16.69,230.02,9.5,0,5,230.02,2005-06-08,20,3
737,738,1,2025-07-10 09:54:18,28289,28289,CO,bank,mobile,US,0,...,12.99,4.44,81.96,4.4,0,1,81.96,2005-06-08,20,4


## 2. Data Understanding

In [3]:
print(df.info())
print(df["is_fraud"].value_counts())
print("Fraud rate:", df["is_fraud"].mean())

# Quick profile stats from reusable chapter utilities
profile_cols = [c for c in ["is_fraud", "num_items", "total_value", "avg_weight", "customer_age"] if c in df.columns]
display(unistats(df[profile_cols], viz=False))

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 592 to 4998
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   order_id              5000 non-null   int64         
 1   customer_id           5000 non-null   int64         
 2   order_datetime        5000 non-null   datetime64[ns]
 3   billing_zip           5000 non-null   object        
 4   shipping_zip          5000 non-null   object        
 5   shipping_state        5000 non-null   object        
 6   payment_method        5000 non-null   object        
 7   device_type           5000 non-null   object        
 8   ip_country            5000 non-null   object        
 9   promo_used            5000 non-null   int64         
 10  promo_code            1261 non-null   object        
 11  order_subtotal        5000 non-null   float64       
 12  shipping_fee          5000 non-null   float64       
 13  tax_amount           

,Count,Unique,Type,Min,Max,25%,50%,75%,Mean,Median,Mode,Std,Skew,Kurt
is_fraud,5000,2,int64,0.00,1.00,0.00,0.00,0.00,0.06,0.00,0.00,0.24,3.58,10.80
num_items,5000,12,int64,1.00,12.00,2.00,4.00,6.00,4.11,4.00,5.00,2.21,0.40,-0.51
total_value,5000,4857,float64,5.38,2053.11,185.76,364.84,596.94,421.55,364.84,38.81,305.18,1.05,1.35
customer_age,5000,54,Int64,18.00,75.00,20.00,24.50,36.00,31.15,24.50,20.00,14.39,1.23,0.21


## 3. Data Preparation

In [4]:
df = df.dropna(subset=["is_fraud"]).copy()

# 1) Basic cleanup: standardize/drop low-value columns
df = basic_wrangling(df, messages=False)

# 2) Parse known date columns into reusable temporal features
date_cols = [c for c in ["order_datetime", "birthdate"] if c in df.columns]
if date_cols:
    df = parse_date(df, features=date_cols, drop_date=False, messages=False)

# 3) Create date-difference signal (birthdate -> order datetime)
if "birthdate" in df.columns and "order_datetime" in df.columns:
    df = manage_dates(
        df,
        startdate="birthdate",
        enddate="order_datetime",
        retain_original=True,
        show_details=False,
    )

# total_value comes from order_total in the load cell; guard if you skip that cell
if "total_value" not in df.columns and "order_total" in df.columns:
    df["total_value"] = df["order_total"]

df["order_value_per_item"] = df["total_value"] / (df["num_items"] + 1)

# If customer_age is unavailable, approximate from days_between
if "customer_age" not in df.columns and "days_between" in df.columns:
    df["customer_age"] = (df["days_between"] / 365).round().astype("Int64")

features = [
    "num_items",
    "total_value",
    "avg_weight",
    "customer_age",
    "customer_order_count",
    "order_value_per_item",
    "order_datetime_year",
    "order_datetime_month",
    "order_datetime_hour",
    "days_between",
]
features = [c for c in features if c in df.columns]

X = df[features]
y = df["is_fraud"].astype(int)
print("Training features:", features)

Training features: ['num_items', 'total_value', 'customer_age', 'customer_order_count', 'order_value_per_item', 'order_datetime_year', 'order_datetime_month', 'order_datetime_hour', 'days_between']


c:\Users\peter\OneDrive - Brigham Young University\Desktop\455Chapter17\crispdm-pipeline-model\functions.py:498: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(out[col], errors='coerce')
c:\Users\peter\OneDrive - Brigham Young University\Desktop\455Chapter17\crispdm-pipeline-model\functions.py:498: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(out[col], errors='coerce')
c:\Users\peter\OneDrive - Brigham Young University\Desktop\455Chapter17\crispdm-pipeline-model\functions.py:498: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  p

## 4. Modeling

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)

pre = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), features)
])

models = {
    "logreg": LogisticRegression(max_iter=1000,class_weight="balanced"),
    "rf": RandomForestClassifier(n_estimators=200,class_weight="balanced")
}

pipelines = {}

for name, m in models.items():
    pipe = Pipeline([("pre",pre),("model",m)])
    pipe.fit(X_train,y_train)
    pipelines[name] = pipe
    print(f"{name} trained")

logreg trained
rf trained


## 5. Evaluation

In [6]:
from sklearn.metrics import classification_report, roc_auc_score

for name, pipe in pipelines.items():
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:,1]
    print(f"\n{name}")
    print(classification_report(y_test,y_pred))
    print("ROC AUC:", roc_auc_score(y_test,y_prob))


logreg
              precision    recall  f1-score   support

           0       0.98      0.69      0.81      1171
           1       0.14      0.77      0.24        79

    accuracy                           0.70      1250
   macro avg       0.56      0.73      0.53      1250
weighted avg       0.93      0.70      0.77      1250

ROC AUC: 0.7724761915056912

rf
              precision    recall  f1-score   support

           0       0.94      1.00      0.97      1171
           1       0.00      0.00      0.00        79

    accuracy                           0.94      1250
   macro avg       0.47      0.50      0.48      1250
weighted avg       0.88      0.94      0.91      1250

ROC AUC: 0.664821801122053


## 6. Feature Importance

In [7]:
importances = pipelines['rf'].named_steps['model'].feature_importances_
for f, imp in zip(features, importances):
    print(f, imp)

num_items 0.07600982100846419
total_value 0.22726271932988953
customer_age 0.05825139905030905
customer_order_count 0.1352676867715502
order_value_per_item 0.17282744058415883
order_datetime_year 0.0024924084716364267
order_datetime_month 0.05875856862310461
order_datetime_hour 0.10941531796242929
days_between 0.15971463819845794


## 7. Deployment

In [8]:
import joblib
from pathlib import Path

pipeline = pipelines["rf"]

model_path = Path("fraud_model.sav")
joblib.dump(pipeline, model_path)
print(f"Saved: {model_path.resolve()}")


Saved: C:\Users\peter\OneDrive - Brigham Young University\Desktop\455Chapter17\crispdm-pipeline-model\fraud_model.sav
